In [1]:
import pickle

import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [2]:
with open("../artifacts/features.pkl", "rb") as f:
    features = pickle.load(f)
with open("../artifacts/mapping.pkl", "rb") as f:
    mapping = pickle.load(f)
with open("../artifacts/tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

In [3]:
all_captions = [c for caps in mapping.values() for c in caps]

In [4]:
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(c.split()) for c in all_captions)

In [5]:
print(f"vocab_size : {vocab_size}")
print(f"max_length : {max_length}")

vocab_size : 8781
max_length : 37


In [6]:
image_ids = list(mapping.keys())

In [7]:
train_ids, test_ids = train_test_split(image_ids, test_size=0.1, random_state=42)

In [11]:
print(len(train_ids), len(test_ids))

7281 810



## Why a generator instead of one big array?

The whole dataset has ~300,000 of these pairs. Storing every padded input plus a
one-hot target (⁠ 300000 x vocab_size ⁠) would need tens of GB of RAM. Instead,
*05_data_generator* produces these pairs in small batches on the fly during
training.

In [12]:
def data_generator(data_keys, mapping, features, tokenizer, max_length, vocab_size, batch_size):
    X1, X2, y = [], [], []
    n = 0
    while True:
        for key in data_keys:
            captions = mapping[key]
            for caption in captions:
                seq = tokenizer.texts_to_sequences([caption])[0]
                for i in range(1, len(seq)):
                    in_seq = pad_sequences([seq[:i]], maxlen=max_length)[0]
                    out_seq = to_categorical([seq[i]], num_classes=vocab_size)[0]
                    X1.append(features[key][0])
                    X2.append(in_seq)
                    y.append(out_seq)
            n += 1
            if n == batch_size:
                yield {"image": np.array(X1), "text": np.array(X2)}, np.array(y)
                X1, X2, y = [], [], []
                n = 0

In [13]:
batch_size = 32

In [14]:
steps_per_epoch = len(train_ids) // batch_size

In [15]:
steps_per_epoch

227

In [16]:
peek = data_generator(train_ids, mapping, features, tokenizer, max_length, vocab_size, batch_size=2)

In [18]:
inputs, outputs = next(peek)